# Redrob Candidate Ranker Sandbox

This Google Colab notebook acts as a reproducible sandbox for the Redrob candidate ranker. It installs the necessary packages, downloads the candidate ranker repository and the 50-candidate sample data, performs data formatting, executes the ranker offline, and displays the top candidates alongside validation checks.

## 1. Install Dependencies

First, we install the CPU-only dependencies required for feature extraction and model inference. The model utilizes `numpy`, `scikit-learn`, and `xgboost` (falls back to LightGBM or Random Forest if needed).

In [ ]:
# Install required libraries
!pip install -q numpy scikit-learn xgboost pandas

## 2. Prepare Code and Dataset

We clone the candidate ranker repository to retrieve `rank.py`, `validate_submission.py`, and `sample_candidates.json`.

In [ ]:
import os

# Update this URL to point to your repository
REPO_URL = "https://github.com/YOUR_USERNAME/redrob-candidate-ranker.git"
REPO_NAME = "redrob-candidate-ranker"

if not os.path.exists(REPO_NAME):
    print(f"Cloning {REPO_URL}...")
    !git clone {REPO_URL}
    %cd {REPO_NAME}
else:
    print(f"{REPO_NAME} already exists, pulling changes...")
    %cd {REPO_NAME}
    !git pull

print(f"Current working directory: {os.getcwd()}")
print("Workspace files:", os.listdir("."))

## 3. Convert Sample Dataset to JSONL

The candidate ranker is optimized to stream large candidate pools line-by-line to maintain a small memory footprint, which requires the `.jsonl` (JSON Lines) format. We convert the `sample_candidates.json` array file into `sample_candidates.jsonl`.

In [ ]:
import json

input_path = 'sample_candidates.json'
output_path = 'sample_candidates.jsonl'

if not os.path.exists(input_path):
    print(f"ERROR: Could not find {input_path} in the current directory!")
    print("Please verify you cloned the repo successfully and are in the correct workspace.")
else:
    with open(input_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    with open(output_path, 'w', encoding='utf-8') as f:
        for record in data:
            f.write(json.dumps(record) + '\n')
            
    print(f"Successfully converted {len(data)} candidates from JSON to JSONL format.")

## 4. Run the Candidate Ranker

Next, we run the CPU-only offline ranker. It reads the candidates, extracts compact features, performs training/validation, and outputs a ranked CSV file with natural language reasoning for each candidate.

In [ ]:
# Run the ranker. We specify --top-n 100 to conform to the submission criteria,
# but since the sample set contains only 50 candidates, the output file will contain 50 rows.
!python rank.py --candidates sample_candidates.jsonl --out submission_sample.csv --verbose

## 5. View Top Ranked Candidates

Let's display the top 10 candidates selected by the ranker model, including their model scores and the generated reasoning.

In [ ]:
import pandas as pd

if os.path.exists('submission_sample.csv'):
    df = pd.read_csv('submission_sample.csv')
    print(f"Ranked {len(df)} candidates. Showing Top 10:")
    pd.set_option('display.max_colwidth', None)
    display(df.head(10))
else:
    print("ERROR: submission_sample.csv was not generated. Check the ranker output above.")

## 6. Run Formatting Checks

We can execute `validate_submission.py` to ensure the CSV structure matches the guidelines. 

*Note: Because the sandbox runs on a small sample of 50 candidates, the validation script will print a warning/error about the row count not being exactly 100. This is expected and normal for the sandbox demonstration.*

In [ ]:
# Validate submission file structure
!python validate_submission.py submission_sample.csv